In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Load the data
train_data = pd.read_csv('bugs-train.csv')
test_data = pd.read_csv('bugs-test.csv')

# Mapping severity levels to numbers
severity_mapping = {
    'enhancement': 1, 'minor': 2, 'normal': 3, 'major': 4, 'blocker': 5, 'critical': 6
}
train_data['severity'] = train_data['severity'].map(severity_mapping)

# Drop rows with NaN severity values
train_data.dropna(subset=['severity'], inplace=True)

# Text preprocessing
def preprocess_text(text):
    tokens = nltk.word_tokenize(text.lower())
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens if token.isalpha() and token not in stopwords.words('english')]
    return ' '.join(lemmatized_tokens)

train_data['processed_summary'] = train_data['summary'].apply(preprocess_text)
test_data['processed_summary'] = test_data['summary'].apply(preprocess_text)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=3000)
X_train = vectorizer.fit_transform(train_data['processed_summary'])
y_train = train_data['severity']
X_test = vectorizer.transform(test_data['processed_summary'])

# Train the SVM model
model = SVC(kernel='linear', random_state=42)
model.fit(X_train, y_train)

# Predict the severities
predicted_severities = model.predict(X_test)

# Reverse mapping from numbers to severity levels
reverse_severity_mapping = {v: k for k, v in severity_mapping.items()}
predicted_severities = [reverse_severity_mapping[severity] for severity in predicted_severities]

# Prepare submission
submission = pd.DataFrame({
    'bug_id': test_data['bug_id'],
    'severity': predicted_severities
})
submission.to_csv('submission36.csv', index=False)

# If needed: Evaluate the model (use split or cross-validation on training data)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
val_predictions = model.predict(X_val_split)
print("Macro Precision:", precision_score(y_val_split, val_predictions, average='macro'))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Macro Precision: 0.5556277987599442


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
